# ⚡ Nouveaux transitoires extragalactiques — Fink/LSST

Ce notebook est dédié au tag **`extragalactic_new_candidate`** qui sélectionne
les alertes dont la **première détection date de moins de 48h**.
L'objectif est d'identifier rapidement les nouveaux transitoires
pour un éventuel suivi spectroscopique.

Pour chaque objet :
- **Âge** depuis la première détection (en heures)
- **Courbe de lumière précoce** (flux et magnitude AB)
- **Cutouts** Science / Template / Difference
- **Scores Fink** disponibles
- Carte de localisation (Mollweide)
- Tableau récapitulatif avec liens portail

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/tags` · `/api/v1/sources` · `/api/v1/cutouts` · `/api/v1/schema`

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
N_ALERTS     = 50          # Nombre d'alertes à récupérer
STARTDATE    = None        # None = défaut API (alertes les plus récentes)
STOPDATE     = None
BASE_URL     = "https://api.lsst.fink-portal.org"
TAG          = "extragalactic_new_candidate"

# Seuil d'âge en heures — afficher seulement les objets plus récents
# None = pas de filtre sur l'âge
MAX_AGE_HOURS = None

# Magnitude limite pour les "objets brillants" (pour trier le tableau)
BRIGHT_MAG_LIMIT = 21.0

# Affichage
SHOW_CUTOUTS  = True
CUTOUT_CMAP   = 'hot'
CUTOUT_FORMAT = 'PNG'
CUTOUT_KINDS  = ['Science', 'Template', 'Difference']

SAVE_FIGS  = True
OUTPUT_DIR = "new_transients_outputs"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
from io import BytesIO
from PIL import Image
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from IPython.display import display as ipy_display, HTML
from tqdm.notebook import tqdm
from astropy.coordinates import SkyCoord
import astropy.units as u

warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)


mpl.rcParams.update({
    'font.size'         : 11,
    'axes.titlesize'    : 16,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 16,
    'xtick.labelsize'   : 16,
    'ytick.labelsize'   : 16,
    'legend.fontsize'   : 16,
    'figure.titlesize'  : 13,
    'figure.titleweight': 'bold',
})

FILTER_COLORS = {
    'u': '#7B2FBE',
    'g': '#0077BB',
    'r': '#33AA77',
    'i': '#DDAA33',
    'z': '#BB5500',
    'y': '#AA0000',
}

MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')
ZP_AB     = 2.5 * np.log10(3631e9)

def mjd_to_datetime(s):
    return MJD_EPOCH + pd.to_timedelta(pd.to_numeric(s, errors='coerce'), unit='D')

def mjd_to_datestr(m):
    try: return (MJD_EPOCH + pd.to_timedelta(float(m), unit='D')).strftime('%Y-%m-%d %H:%M')
    except: return str(m)

def mjd_now():
    """MJD approximatif de maintenant (UTC)."""
    return (pd.Timestamp.utcnow() - MJD_EPOCH).total_seconds() / 86400.0

def flux_to_mag(flux, flux_e):
    flux, flux_e = np.asarray(flux, float), np.asarray(flux_e, float)
    v = flux > 0
    mag = np.full(flux.shape, np.nan)
    me  = np.full(flux.shape, np.nan)
    mag[v] = -2.5 * np.log10(flux[v]) + ZP_AB
    me[v]  = (2.5 / np.log(10)) * np.abs(flux_e[v] / flux[v])
    return mag, me

def radec_to_mollweide(ra, dec):
    return -np.deg2rad(np.asarray(ra) - 180.), np.deg2rad(np.asarray(dec))

def split_curve(x, y):
    breaks = np.where(np.abs(np.diff(x)) > np.pi * 0.8)[0] + 1
    return np.split(x, breaks), np.split(y, breaks)

print("Imports OK ✓")

## 3. Colonnes disponibles (scores clf)

In [ ]:
# Schéma — colonnes clf
# /api/v1/schema retourne un dict à deux sections :
#   "LSST original fields (r:)" : champs originaux LSST
#   "Fink added values (f:)"    : valeurs ajoutées par Fink
resp = requests.post(f"{BASE_URL}/api/v1/schema",
                     json={"endpoint": "/api/v1/sources"}, timeout=60)
resp.raise_for_status()
_schema = resp.json()
_fink_section = next(
    (v for k, v in _schema.items() if 'fink' in k.lower() or k.startswith('f:')),
    {}
)
clf_cols = [f"f:{k}" for k in _fink_section if k.startswith('clf')]
print(f"Scores clf disponibles ({len(clf_cols)}) : {clf_cols}")


## 4. Récupération des alertes et calcul de l'âge

In [ ]:
# ── Alertes via /tags ──────────────────────────────────────────────────────────
payload = {"tag": TAG, "n": N_ALERTS, "output-format": "json"}
if STARTDATE: payload["startdate"] = STARTDATE
if STOPDATE:  payload["stopdate"]  = STOPDATE
r = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
r.raise_for_status()
df_tags = pd.DataFrame(r.json())
df_tags.columns = [c.replace('r:','').replace('f:','') for c in df_tags.columns]
id_col = next(c for c in df_tags.columns if 'diaObjectId' in c)
object_ids = df_tags[id_col].astype(str).unique().tolist()
print(f"✓ {len(df_tags)} alertes, {len(object_ids)} objets uniques\n")

# ── Sources enrichies (courbe + scores) ───────────────────────────────────────
base_cols  = "r:diaObjectId,r:diaSourceId,r:midpointMjdTai,r:band,r:psfFlux,r:psfFluxErr,r:ra,r:dec"
extra_cols = ','.join(clf_cols[:4]) if clf_cols else ''
all_cols   = base_cols + (',' + extra_cols if extra_cols else '')

records = []
for oid in tqdm(object_ids, desc='Récupération sources', unit='obj'):
    try:
        r2 = requests.post(f"{BASE_URL}/api/v1/sources",
                           json={"diaObjectId": str(oid), "columns": all_cols,
                                 "output-format": "json"}, timeout=60)
        r2.raise_for_status()
        if r2.json():
            df_src = pd.DataFrame(r2.json())
            df_src.columns = [c.replace('r:','').replace('f:','') for c in df_src.columns]
            records.append(df_src)
    except Exception as e:
        print(f"  ✗ {oid} : {e}")

df_all = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
for col in ['midpointMjdTai','psfFlux','psfFluxErr','ra','dec']:
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

# ── Calcul de l'âge depuis la première détection ──────────────────────────────
mjd_now_val = mjd_now()
first_mjd   = df_all.groupby('diaObjectId')['midpointMjdTai'].min()
age_hours   = ((mjd_now_val - first_mjd) * 24).rename('age_hours')
df_age      = age_hours.reset_index()

# Filtre optionnel sur l'âge
if MAX_AGE_HOURS:
    df_age = df_age[df_age['age_hours'] <= MAX_AGE_HOURS]
    object_ids = df_age['diaObjectId'].astype(str).tolist()
    df_all = df_all[df_all['diaObjectId'].astype(str).isin(object_ids)]
    print(f"\nAprès filtre âge ≤ {MAX_AGE_HOURS}h : {len(object_ids)} objets")

# Trier par âge croissant (les plus récents d'abord)
df_age = df_age.sort_values('age_hours')
object_ids_sorted = df_age['diaObjectId'].astype(str).tolist()

print(f"\nÂge min : {df_age['age_hours'].min():.1f} h")
print(f"Âge max : {df_age['age_hours'].max():.1f} h")
print(f"\n{len(object_ids_sorted)} objets chargés.")

## 5. Distribution des âges et magnitude au pic

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout='constrained')
fig.suptitle(f"Propriétés des nouveaux transitoires — {TAG}")

# Âge
axes[0].hist(df_age['age_hours'], bins=20, color='#4477AA',
             edgecolor='white', linewidth=0.4, alpha=0.85)
axes[0].axvline(48, color='#EE6677', lw=1.5, ls='--', label='48h (critère tag)')
axes[0].set_xlabel('Âge depuis 1ère détection (h)')
axes[0].set_ylabel('N objets')
axes[0].set_title('Distribution des âges')
axes[0].legend(fontsize=8)
axes[0].grid(True, axis='y', alpha=0.25, linewidth=0.5)

# Flux max par objet → magnitude
flux_max = df_all.groupby('diaObjectId')['psfFlux'].max()
flux_max = flux_max[flux_max > 0]
mag_max  = -2.5 * np.log10(flux_max.values) + ZP_AB
axes[1].hist(mag_max, bins=20, color='#228833',
             edgecolor='white', linewidth=0.4, alpha=0.85)
axes[1].axvline(BRIGHT_MAG_LIMIT, color='#EE6677', lw=1.5, ls='--',
                label=f'Limite {BRIGHT_MAG_LIMIT} mag')
axes[1].set_xlabel('Magnitude AB au pic')
axes[1].set_ylabel('N objets')
axes[1].set_title('Magnitude au pic (flux max)')
axes[1].legend(fontsize=8)
axes[1].grid(True, axis='y', alpha=0.25, linewidth=0.5)

# Filtres
if 'band' in df_all.columns:
    vc = df_all['band'].value_counts().sort_index()
    colors_f = [FILTER_COLORS.get(b,'#888') for b in vc.index]
    axes[2].bar(vc.index, vc.values, color=colors_f,
                edgecolor='white', linewidth=0.5)
    axes[2].set_xlabel('Filtre')
    axes[2].set_ylabel('N alertes')
    axes[2].set_title('Alertes par filtre')
    axes[2].grid(True, axis='y', alpha=0.25, linewidth=0.5)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/new_transients_stats.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/new_transients_stats.png", bbox_inches='tight', dpi=150)
    print("✓ Sauvegardé")
plt.show()

## 6. Carte Mollweide — colorée par âge

In [ ]:
df_pos = df_all.groupby('diaObjectId').agg(
    ra=('ra','median'), dec=('dec','median')).reset_index()
df_pos = df_pos.merge(df_age[['diaObjectId','age_hours']], on='diaObjectId', how='left')
df_pos = df_pos.dropna(subset=['ra','dec'])

# Plan galactique
l = np.linspace(0, 360, 1000)
gc = SkyCoord(l=l*u.deg, b=np.zeros(1000)*u.deg, frame='galactic').icrs
idx = np.argsort(gc.ra.deg)
gal_x, gal_y = radec_to_mollweide(gc.ra.deg[idx], gc.dec.deg[idx])

fig = plt.figure(figsize=(16, 8), layout='constrained')
ax  = fig.add_subplot(111, projection='mollweide')
fig.suptitle(f"Localisation — {TAG} ({len(df_pos)} objets)")
ax.grid(True, color='gray', alpha=0.3, linewidth=0.5, linestyle='--')

for sx, sy in zip(*split_curve(gal_x, gal_y)):
    ax.plot(sx, sy, color='goldenrod', lw=1.2)

px, py = radec_to_mollweide(df_pos['ra'].values, df_pos['dec'].values)
sc = ax.scatter(px, py, c=df_pos['age_hours'].values,
                cmap='plasma_r', s=25, vmin=0, vmax=48,
                alpha=0.85, edgecolors='white', linewidths=0.3, zorder=5)
cbar = fig.colorbar(sc, ax=ax, orientation='horizontal',
                    pad=0.04, fraction=0.025, label='Âge (heures)')
ax.plot([], [], 'o', color='yellow', ms=6, label='Très récent (< 6h)')
ax.plot([], [], '-', color='goldenrod', lw=1.2, label='Plan galactique')
ax.legend(loc='lower left', fontsize=8, framealpha=0.75)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/new_transients_skymap.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/new_transients_skymap.png", bbox_inches='tight', dpi=150)
    print("✓ Sauvegardé")
plt.show()

## 7. Figures de synthèse — courbes de lumière précoces + cutouts

Objets triés du plus récent au plus ancien.

In [ ]:
clf_present = [c.replace('f:','') for c in clf_cols if c.replace('f:','') in df_all.columns]

for oid in object_ids_sorted:
    lc = df_all[df_all['diaObjectId'].astype(str) == str(oid)].copy()
    lc = lc.sort_values('midpointMjdTai')
    lc['date'] = mjd_to_datetime(lc['midpointMjdTai'])

    age_h = df_age[df_age['diaObjectId'].astype(str)==str(oid)]['age_hours'].values
    age_h = age_h[0] if len(age_h) > 0 else np.nan

    # Dernière source (pour cutouts)
    lc_sorted   = lc.dropna(subset=['midpointMjdTai']).sort_values('midpointMjdTai', ascending=False)
    last_row    = lc_sorted.iloc[0]
    last_src_id = last_row.get('diaSourceId', None)
    last_band   = last_row.get('band', 'N/A')
    last_date   = mjd_to_datestr(last_row.get('midpointMjdTai'))

    # Score
    score_str = ''
    if clf_present:
        s = pd.to_numeric(lc[clf_present[0]], errors='coerce').median()
        if not np.isnan(s):
            score_str = f"   {clf_present[0].replace('clf.','')} : {s:.3f}"

    # Magnitude pic
    flux_pk = lc['psfFlux'].max()
    mag_str = f"{-2.5*np.log10(flux_pk)+ZP_AB:.2f}" if flux_pk > 0 else '—'

    title = (f"diaObjectId : {oid}   |   "
             f"âge : {age_h:.1f}h   "
             f"mag pic : {mag_str}   "
             f"1ère détection : {mjd_to_datestr(lc['midpointMjdTai'].min())}"
             f"{score_str}")

    # ── Mise en page ──────────────────────────────────────────────────────────
    FIG_W    = 16.0
    CUT_SIZE = FIG_W * 0.90 * 0.94 / 3
    LC_H     = CUT_SIZE * 0.90

    fig = plt.figure(figsize=(FIG_W, LC_H + CUT_SIZE + 1), layout='constrained')
    fig.suptitle(title, fontsize=10, fontweight='bold')

    gs     = GridSpec(2, 1, figure=fig, height_ratios=[LC_H, CUT_SIZE])
    gs_lc  = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[0], wspace=0.07)
    ax_f   = fig.add_subplot(gs_lc[0])
    ax_m   = fig.add_subplot(gs_lc[1])

    gs_bot = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[1],
                                     width_ratios=[0.94, 0.06], wspace=0.02)
    gs_cut = GridSpecFromSubplotSpec(1, 3, subplot_spec=gs_bot[0], wspace=0.03)
    axes_cut = [fig.add_subplot(gs_cut[i]) for i in range(3)]
    ax_cbar  = fig.add_subplot(gs_bot[1])
    ax_cbar.set_visible(False)

    # ── Courbes de lumière ────────────────────────────────────────────────────
    bands_present = sorted(lc['band'].dropna().unique())
    for band in bands_present:
        sub   = lc[lc['band']==band].dropna(subset=['psfFlux','psfFluxErr'])
        color = FILTER_COLORS.get(band,'#888')
        ax_f.errorbar(sub['date'], sub['psfFlux'], yerr=sub['psfFluxErr'],
                      fmt='o-', color=color, label=f"${band}$",
                      ms=5, capsize=3, lw=1.0, alpha=0.85)
        mag_v, me_v = flux_to_mag(sub['psfFlux'].values, sub['psfFluxErr'].values)
        det = ~np.isnan(mag_v)
        if det.any():
            ax_m.errorbar(sub['date'].values[det], mag_v[det], yerr=me_v[det],
                          fmt='o-', color=color, label=f"${band}$",
                          ms=5, capsize=3, lw=1.0, alpha=0.85)

    ax_f.axhline(0, color='gray', lw=0.6, ls='--', alpha=0.5)
    ax_f.set_ylabel('Flux PSF (nJy)')
    ax_f.legend(ncol=len(bands_present), loc='upper left', handlelength=1)
    ax_f.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.setp(ax_f.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax_f.grid(True, alpha=0.2, linewidth=0.5)

    ax_m.invert_yaxis()
    ax_m.set_ylabel('Magnitude AB')
    ax_m.legend(ncol=len(bands_present), loc='upper left', handlelength=1)
    ax_m.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.setp(ax_m.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax_m.grid(True, alpha=0.2, linewidth=0.5)

    # ── Cutouts ───────────────────────────────────────────────────────────────
    if SHOW_CUTOUTS and last_src_id is not None:
        arrays, pairs = [], []
        for ax_c, kind in zip(axes_cut, CUTOUT_KINDS):
            try:
                r3 = requests.get(f"{BASE_URL}/api/v1/cutouts",
                                  params={"diaSourceId": str(last_src_id),
                                          "kind": kind, "output-format": CUTOUT_FORMAT},
                                  timeout=60)
                r3.raise_for_status()
                arr = np.array(Image.open(BytesIO(r3.content))).astype(float)
                if arr.ndim == 3: arr = arr.mean(axis=2)
                arrays.append(arr)
                im = ax_c.imshow(arr, cmap=CUTOUT_CMAP, origin='upper', aspect='auto')
                ax_c.axis('off')
                ax_c.set_title(f"{kind}\n{last_band}  {last_date}", fontsize=9)
                pairs.append((ax_c, im, arr))
            except Exception as e:
                ax_c.axis('off')
                ax_c.text(0.5, 0.5, f"{kind}\nerreur", ha='center', va='center',
                          transform=ax_c.transAxes, fontsize=8, color='red')
        if pairs:
            norm_c = plt.Normalize(vmin=min(a.min() for _,_,a in pairs),
                                   vmax=max(a.max() for _,_,a in pairs))
            fig.canvas.draw()
            pos_d = axes_cut[-1].get_position()
            pos_s = ax_cbar.get_position()
            cax   = fig.add_axes([pos_s.x0+pos_s.width*0.10, pos_d.y0,
                                   pos_s.width*0.35, pos_d.height])
            sm = plt.cm.ScalarMappable(cmap=CUTOUT_CMAP, norm=norm_c)
            sm.set_array([])
            fig.colorbar(sm, cax=cax, label='ADU')
    else:
        for ax_c in axes_cut: ax_c.axis('off')

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/new_{oid}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/new_{oid}.png", bbox_inches='tight', dpi=150)

    plt.show()

    fink_url = f"https://lsst.fink-portal.org/{oid}"
    ipy_display(HTML(
        f'<div style="font-size:12px;margin:2px 0 14px 4px;">'
        f'🔗 <b>Portail Fink/LSST</b> : '
        f'<a href="{fink_url}" target="_blank">{fink_url}</a></div>'
    ))
    print()

print("\n✅ Affichage terminé.")

## 8. Tableau récapitulatif — trié par âge croissant

In [ ]:
agg = {'ra': 'median', 'dec': 'median', 'psfFlux': 'max',
       'midpointMjdTai': ['min','max'],
       'band': lambda x: ', '.join(sorted(x.dropna().unique()))}
for c in clf_present:
    agg[c] = 'median'

df_sum = df_all.groupby('diaObjectId').agg(agg).reset_index()
df_sum.columns = ['_'.join(c).strip('_') for c in df_sum.columns]
df_sum = df_sum.rename(columns={
    'midpointMjdTai_min': 'MJD_first',
    'midpointMjdTai_max': 'MJD_last',
    'psfFlux_max': 'flux_max (nJy)',
    'ra_median': 'RA', 'dec_median': 'Dec',
})
df_sum = df_sum.merge(df_age[['diaObjectId','age_hours']], on='diaObjectId', how='left')
df_sum['1ère détection'] = df_sum['MJD_first'].apply(mjd_to_datestr)
df_sum['dernière alerte'] = df_sum['MJD_last'].apply(mjd_to_datestr)
df_sum['âge (h)'] = df_sum['age_hours'].round(1)
df_sum['flux_max (nJy)'] = df_sum['flux_max (nJy)'].round(1)
df_sum['RA']  = df_sum['RA'].round(5)
df_sum['Dec'] = df_sum['Dec'].round(5)

# Magnitude pic
df_sum['mag_pic'] = df_sum['flux_max (nJy)'].apply(
    lambda f: round(-2.5*np.log10(f)+ZP_AB, 2) if f > 0 else np.nan)

for c in clf_present:
    if c in df_sum.columns:
        df_sum[c] = df_sum[c].round(3)

df_sum = df_sum.sort_values('âge (h)')
df_sum['Portail Fink'] = df_sum['diaObjectId'].apply(
    lambda o: f'<a href="https://lsst.fink-portal.org/{o}" target="_blank">🔗 {o}</a>')

keep = ['diaObjectId', 'RA', 'Dec', 'âge (h)', '1ère détection',
        'dernière alerte', 'flux_max (nJy)', 'mag_pic']
keep += [c for c in clf_present if c in df_sum.columns]
keep += ['band', 'Portail Fink']
keep = [c for c in keep if c in df_sum.columns]

html = df_sum[keep].to_html(index=False, escape=False, border=0, classes='fink-table')
style = """
<style>
  .fink-table { border-collapse:collapse; font-size:12px; width:100%; }
  .fink-table th { background:#f0f0f0; padding:6px 10px;
                   border-bottom:2px solid #ccc; text-align:left; }
  .fink-table td { padding:4px 10px; border-bottom:1px solid #eee;
                   text-align:left; white-space:nowrap; }
  .fink-table tr:hover td { background:#fafafa; }
</style>
"""
ipy_display(HTML(style + html))